<a href="https://colab.research.google.com/github/TanyaGupta37/ML-Lab/blob/main/MERT_MER_Improved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🎵 Music Emotion Recognition (MER) using MERT + Improved Pipeline


In [ ]:
!pip install -q transformers librosa soundfile

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import torch
from tqdm import tqdm

from transformers import AutoModel, AutoProcessor
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score


In [ ]:

MERT_MODEL_NAME = 'm-a-p/MERT-v1-95M'
processor = AutoProcessor.from_pretrained(MERT_MODEL_NAME, trust_remote_code=True)
model = AutoModel.from_pretrained(MERT_MODEL_NAME, trust_remote_code=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/211 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

configuration_MERT.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/m-a-p/MERT-v1-95M:
- configuration_MERT.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
The image processor of type `Wav2Vec2ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
`use_fast` is set to `True` but the image processor class does not have a fast version.  Falling back to the slow version.


modeling_MERT.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/m-a-p/MERT-v1-95M:
- modeling_MERT.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

MERTModel(
  (feature_extractor): HubertFeatureEncoder(
    (conv_layers): ModuleList(
      (0): HubertGroupNormConvLayer(
        (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
        (activation): GELUActivation()
        (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
      )
      (1-4): 4 x HubertNoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
      (5-6): 2 x HubertNoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
    )
  )
  (feature_projection): MERTFeatureProjection(
    (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (projection): Linear(in_features=512, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): HubertEncoder(
    (pos_conv_embed): HubertPositionalConvEmbedding(
      (conv): Parametr

In [ ]:

LABELS_PATH = '/content/drive/MyDrive/emotify_colab/data/emotify_labels_matched.csv'
AUDIO_BASE = '/content/drive/MyDrive/emotify_colab/data/audio/emotify'

SAMPLE_RATE = 24000
SEGMENT_DURATION = 10
SEGMENTS_PER_SONG = 5


In [ ]:

def extract_segment_embedding(audio_path):
    y, sr = librosa.load(audio_path, sr=SAMPLE_RATE, duration=60)
    target_len = SAMPLE_RATE * SEGMENT_DURATION

    if len(y) > target_len:
        start = np.random.randint(0, len(y) - target_len)
        y = y[start:start + target_len]
    else:
        y = np.pad(y, (0, target_len - len(y)))

    inputs = processor(y, sampling_rate=SAMPLE_RATE, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    layer5 = outputs.hidden_states[5].squeeze(0).mean(dim=0).cpu().numpy()
    layer6 = outputs.hidden_states[6].squeeze(0).mean(dim=0).cpu().numpy()

    return np.concatenate([layer5, layer6])


In [ ]:
df = pd.read_csv(LABELS_PATH)

X, y, song_ids = [], [], []

for i, row in tqdm(df.iterrows(), total=len(df)):
    path = os.path.join(AUDIO_BASE, row['filename'])

    if not os.path.exists(path):
        continue

    label = row['dominant_emotion']

    try:
        for _ in range(SEGMENTS_PER_SONG):
            emb = extract_segment_embedding(path)
            X.append(emb)
            y.append(label)
            song_ids.append(i)
    except:
        continue

X = np.array(X)
y = np.array(y)
song_ids = np.array(song_ids)


100%|██████████| 399/399 [2:48:31<00:00, 25.34s/it]


In [ ]:

le = LabelEncoder()
y_enc = le.fit_transform(y)


In [ ]:

unique_songs = np.unique(song_ids)
song_labels = np.array([y_enc[song_ids == s][0] for s in unique_songs])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accs, f1s, aucs = [], [], []

for train_idx, test_idx in skf.split(unique_songs, song_labels):

    train_songs = unique_songs[train_idx]
    test_songs = unique_songs[test_idx]

    train_mask = np.isin(song_ids, train_songs)

    X_train = X[train_mask]
    y_train = y_enc[train_mask]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)

    pca = PCA(n_components=100)
    X_train = pca.fit_transform(X_train)

    model = LogisticRegression(max_iter=1000, class_weight='balanced')
    model.fit(X_train, y_train)

    y_true, y_pred, y_proba = [], [], []

    for s in test_songs:
        mask = song_ids == s
        X_song = scaler.transform(X[mask])
        X_song = pca.transform(X_song)

        probs = model.predict_proba(X_song)
        avg_prob = probs.mean(axis=0)

        pred = np.argmax(avg_prob)

        y_true.append(y_enc[mask][0])
        y_pred.append(pred)
        y_proba.append(avg_prob)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_proba = np.array(y_proba)

    accs.append(accuracy_score(y_true, y_pred))
    f1s.append(f1_score(y_true, y_pred, average='macro'))

    y_bin = label_binarize(y_true, classes=range(len(le.classes_)))
    aucs.append(roc_auc_score(y_bin, y_proba, multi_class='ovr'))

print("Accuracy:", np.mean(accs))
print("F1:", np.mean(f1s))
print("AUC:", np.mean(aucs))


Accuracy: 0.13034810126582277
F1: 0.11949821935790436
AUC: 0.5185001116124025
